# 08 — Pixel Histograms and Minkowski Functionals

**Purpose:** Evaluate one-point statistics and morphological properties of DDPM samples
against Agora and Gaussian baselines.

This notebook covers two complementary non-Gaussian diagnostics:

1. **Pixel intensity histograms** — normalised histograms of pixel values for 200 CIB
   and tSZ samples from each source (Agora, DDPM, Gaussian), binned over [0, 100] μK
   for CIB and [−100, 0] μK for tSZ in 1000 bins and smoothed with a Gaussian kernel
   (σ = 1). Probes the one-point PDF and highlights the DDPM's underproduction of
   extreme-value pixels.

2. **Minkowski functionals** — computes M0 (area fraction), M1 (total boundary length),
   and M2 (Euler characteristic) of excursion sets across 50 intensity thresholds
   ν ∈ [0, 1] using the Boelens & Tchelepi software package. Applied to min-max
   normalised maps for CIB and tSZ separately.

**Inputs:**
- Test maps and DDPM samples: `data/low_pass/2mJy/*.npy`
- Gaussian baseline: `data/low_pass/2mJy/gaussian_cib_tsz_2mJy_lp.npy`

**Outputs:** pixel histogram plots (Figure 5), Minkowski functional plots (Figure 6).

**Key module functions:**
- `foregrounds_diffusion.preprocessing.apply_maxmin_normalization`
- `foregrounds_diffusion.preprocessing.apply_stdnorm`

**External dependency:** `minkowski` (Boelens & Tchelepi package) for Minkowski
functional computation.

**Paper reference:** §3.2 (statistics definitions), §4.4 (Figure 5), §4.5 (Figure 6).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

from foregrounds_diffusion.preprocessing import apply_maxmin_normalization, apply_stdnorm

PTSRC     = 2
N_SAMPLES = 200   # number of maps used per source


In [ ]:
cib_maps   = np.load(f"data/low_pass/{PTSRC}mJy/CIB_map_150GHz_256_st6_minmax_{PTSRC}mJy_zero_lp.npy")
tsz_maps   = np.load(f"data/low_pass/{PTSRC}mJy/tSZ3_map_150GHz_256_st6_minmax_{PTSRC}mJy_norm_lp.npy")
ddpm_raw   = np.load(f"data/low_pass/{PTSRC}mJy/new_samples_cib_tsz_{PTSRC}mJy_lp.npy")
gauss_maps = np.load(f"data/low_pass/{PTSRC}mJy/gaussian_cib_tsz_{PTSRC}mJy_lp.npy")

# Collect pixel values from N_SAMPLES maps per source
agora_cib_px  = cib_maps[:N_SAMPLES, :, :, 0].ravel()
agora_tsz_px  = tsz_maps[:N_SAMPLES, :, :, 0].ravel()
ddpm_cib_px   = ddpm_raw[:N_SAMPLES, 0].ravel()
ddpm_tsz_px   = ddpm_raw[:N_SAMPLES, 1].ravel()
if gauss_maps.shape[1] == 2:
    gauss_cib_px = gauss_maps[:N_SAMPLES, 0].ravel()
    gauss_tsz_px = gauss_maps[:N_SAMPLES, 1].ravel()
else:
    gauss_cib_px = gauss_maps[:N_SAMPLES, :, :, 0].ravel()
    gauss_tsz_px = gauss_maps[:N_SAMPLES, :, :, 1].ravel()


In [ ]:
# Pixel intensity histograms (paper Figure 5)
# CIB is in [0,1] after min-max; tSZ is std-normalised (mean 0, std 1)
N_BINS   = 1000
SMOOTH_S = 1.0   # Gaussian smoothing sigma (bins)

bins_cib = np.linspace(0.,  1.,   N_BINS + 1)
bins_tsz = np.linspace(-5., 5.,   N_BINS + 1)
bc_cib   = 0.5 * (bins_cib[:-1] + bins_cib[1:])
bc_tsz   = 0.5 * (bins_tsz[:-1] + bins_tsz[1:])

def hist_smooth(px, bins, sigma):
    h, _ = np.histogram(px, bins=bins, density=True)
    return gaussian_filter1d(h, sigma=sigma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for px, label, color in [
    (agora_cib_px, "Agora",    "C0"),
    (ddpm_cib_px,  "DDPM",     "C1"),
    (gauss_cib_px, "Gaussian", "C2"),
]:
    ax1.plot(bc_cib, hist_smooth(px, bins_cib, SMOOTH_S), label=label, color=color)
ax1.set_xlabel("CIB pixel value (normalised)");  ax1.set_ylabel("PDF");  ax1.legend()
ax1.set_yscale("log")

for px, label, color in [
    (agora_tsz_px, "Agora",    "C0"),
    (ddpm_tsz_px,  "DDPM",     "C1"),
    (gauss_tsz_px, "Gaussian", "C2"),
]:
    ax2.plot(bc_tsz, hist_smooth(px, bins_tsz, SMOOTH_S), label=label, color=color)
ax2.set_xlabel("tSZ pixel value (std-normalised)");  ax2.legend()
ax2.set_yscale("log")

plt.tight_layout();  plt.show()


In [ ]:
try:
    import minkowski
    HAS_MINKOWSKI = True
except ImportError:
    HAS_MINKOWSKI = False
    print("minkowski package not found — install from Boelens & Tchelepi (2021).")
    print("Skipping Minkowski functional computation.")

N_THRESH   = 50
thresholds = np.linspace(0., 1., N_THRESH)

def compute_mfs(maps_nhw, norm_fn, thresholds):
    """Return mean M0, M1, M2 across N maps at each threshold."""
    M0 = np.zeros((len(maps_nhw), len(thresholds)))
    M1 = np.zeros_like(M0)
    M2 = np.zeros_like(M0)
    for i, m in enumerate(maps_nhw):
        m_norm = norm_fn(m)
        for t, thresh in enumerate(thresholds):
            binary = (m_norm > thresh).astype(float)
            if HAS_MINKOWSKI:
                M0[i, t] = minkowski.m0(binary)
                M1[i, t] = minkowski.m1(binary)
                M2[i, t] = minkowski.m2(binary)
            else:
                # Fallback: area fraction only
                M0[i, t] = binary.mean()
    return M0, M1, M2


In [ ]:
if HAS_MINKOWSKI:
    # Apply min-max normalisation before thresholding (maps must be on a common scale)
    norm = lambda m: apply_maxmin_normalization(m)
    N_MF = min(N_SAMPLES, 50)   # MF computation is slow; use a subset

    M0_agora, M1_agora, M2_agora = compute_mfs(cib_maps[:N_MF, :, :, 0], norm, thresholds)
    M0_ddpm,  M1_ddpm,  M2_ddpm  = compute_mfs(ddpm_raw[:N_MF, 0],       norm, thresholds)
    M0_gauss, M1_gauss, M2_gauss = compute_mfs(gauss_cib[:N_MF] if gauss_maps.shape[1] == 2
                                                else gauss_maps[:N_MF, :, :, 0], norm, thresholds)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    titles = ["M0 (area fraction)", "M1 (perimeter)", "M2 (Euler characteristic)"]
    for ax, M_a, M_d, M_g, title in zip(
        axes,
        [M0_agora, M1_agora, M2_agora],
        [M0_ddpm,  M1_ddpm,  M2_ddpm],
        [M0_gauss, M1_gauss, M2_gauss],
        titles,
    ):
        ax.plot(thresholds, M_a.mean(axis=0), label="Agora",    color="C0")
        ax.plot(thresholds, M_d.mean(axis=0), label="DDPM",     color="C1")
        ax.plot(thresholds, M_g.mean(axis=0), label="Gaussian", color="C2", ls="--")
        ax.set_xlabel(r"Threshold $\nu$");  ax.set_title(title);  ax.legend()
    plt.tight_layout();  plt.show()
